# Practice Lab: Deep Research Agents (Lesson 11.8)

Module 11 . 8 exercises . Decompose + search + map-reduce + credibility + citations + caching + India

Exercises 1-5 run locally (pure Python).


# Lesson 11.8: Deep Research Agents

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design.


In [ ]:
import json
import math
import numpy as np
from datetime import datetime, timedelta
np.random.seed(42)


---
## Exercise 1 (Easy): Query Decomposer



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def decompose(q):
    templates = {"worth":[{"q":"What does the curriculum cover?","dep":None},{"q":"How does pricing compare?","dep":None},
        {"q":"What career outcomes?","dep":None},{"q":"Prerequisites and time?","dep":None},{"q":"ROI calculation?","dep":"pricing"}],
        "trends":[{"q":"Popular agent frameworks 2026?","dep":None},{"q":"Company deployments?","dep":None},
        {"q":"Safety concerns?","dep":None},{"q":"Benchmark progress?","dep":None}],
        "compare":[{"q":"Product A features?","dep":None},{"q":"Product B features?","dep":None},
        {"q":"Price comparison?","dep":None},{"q":"Reviews?","dep":None},{"q":"Recommendation?","dep":"all"}]}
    for k,subs in templates.items():
        if k in q.lower(): return subs
    return [{"q":q,"dep":None}]

print("Query Decomposer:")
total=0
for q in ["Is GenAI course worth it?","Current trends in agents?","Compare CrewAI vs LangGraph?"]:
    subs=decompose(q); indep=[s for s in subs if s["dep"] is None]; dep=[s for s in subs if s["dep"]]
    total+=len(subs)
    print(f"  '{q[:35]}' -> {len(subs)} sub-queries ({len(indep)} parallel + {len(dep)} sequential)")
print(f"\n  Total: {total} queries. Key: analyze dependencies BEFORE parallelizing")


</details>


---
## Exercise 2 (Easy): Tavily + Jina Pipeline



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def est_tok(t): return int(len(t.split())*1.3)

def sim_tavily(q):
    return [{"url":"netsetos.com","title":"GenAI Course","snippet":"14999 INR, 146hrs","score":0.92},
            {"url":"linkedin.com","title":"GenAI Careers","snippet":"15-40 LPA","score":0.85}]

def sim_jina(url):
    pages={"netsetos.com":{"raw":2800,"md":"# GenAI Course\n14999 INR | 146hrs | 14 modules | 4.8 stars"},
           "linkedin.com":{"raw":3500,"md":"# GenAI Careers\n15-40 LPA India | 300% growth"}}
    domain=url.split("//")[1].split("/")[0] if "//" in url else url
    return pages.get(domain,{"raw":2000,"md":"Content extracted"})

print("Tavily + Jina Pipeline:")
results=sim_tavily("GenAI course")
for r in results: print(f"  [{r['score']}] {r['title']}: {r['snippet']}")

total_raw=total_md=0
for r in results:
    j=sim_jina(r["url"]); md_tok=est_tok(j["md"])
    total_raw+=j["raw"]; total_md+=md_tok
    print(f"  {r['url']}: {j['raw']} raw -> {md_tok} md ({(1-md_tok/j['raw'])*100:.0f}% savings)")

print(f"\n  APIs: Tavily $0.005 | Serper+Jina $0.0003 | Brave $0.005 | Exa $0.007")
print(f"  Token savings: {total_raw} -> {total_md} = {(1-total_md/total_raw)*100:.0f}% reduction")


</details>


---
## Exercise 3 (Medium): Map-Reduce Summarizer



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)

sources=[{"name":"netsetos.com","tokens":800,"text":"GenAI: 14999 INR, 146hrs, 4.8 stars"},
    {"name":"linkedin.com","tokens":600,"text":"Salaries: 15-40 LPA, 300% growth"},
    {"name":"coursera.org","tokens":700,"text":"Alternative: $294 (24500 INR), 6 months"},
    {"name":"reddit.com","tokens":500,"text":"Practical projects, live classes help"}]

print("Map-Reduce Summarizer:")
map_tok=0
for s in sources:
    st=int(s["tokens"]*0.3); faith=round(0.95-0.05*(1-st/s["tokens"]),2); map_tok+=st
    print(f"  MAP: {s['name']:<16} {s['tokens']:>4}->{st:>3} tok  faith={faith}")

reduce_tok=int(map_tok*0.5); reduce_faith=round(0.90*0.92,2)
print(f"  REDUCE: {map_tok}->{reduce_tok} tok  faith={reduce_faith}")
print(f"\n  Cascade: original(1.00) -> map(0.90) -> reduce({reduce_faith})")
print(f"  ACL 2025: r=-0.685 (more abstract = less faithful)")
print(f"  Fix: quote critical facts, paraphrase synthesis")


</details>


---
## Exercise 4 (Medium): Source Credibility Scorer



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import math

def domain_tier(d):
    for tier,doms in {1:[".gov",".edu","arxiv","nature"],2:["reuters","bbc","techcrunch"],
        3:["medium","linkedin","substack"],4:["reddit","quora"],5:["twitter","tiktok"]}.items():
        if any(x in d.lower() for x in doms): return tier
    return 3

def cred_score(src):
    t=domain_tier(src["domain"]); ts=(6-t)/5
    rec=math.exp(-0.693*src["days"]/180)
    return round(ts*0.4+rec*0.3+src["rel"]*0.3, 2)

sources=[{"domain":"arxiv.org","days":30,"rel":0.9,"title":"Agent Survey"},
    {"domain":"netsetos.com","days":7,"rel":0.95,"title":"Official page"},
    {"domain":"reddit.com","days":14,"rel":0.7,"title":"Student review"},
    {"domain":"techcrunch.com","days":60,"rel":0.6,"title":"AI market"},
    {"domain":"twitter.com","days":2,"rel":0.4,"title":"Random tweet"},
    {"domain":"nature.com","days":365,"rel":0.8,"title":"AI study (old)"}]

print("Source Credibility Scorer:")
scored=[(s,cred_score(s)) for s in sources]
scored.sort(key=lambda x:-x[1])
for i,(s,sc) in enumerate(scored,1):
    flag=" [UNRELIABLE]" if sc<0.4 else ""
    print(f"  {i}. {s['title']:<22} T{domain_tier(s['domain'])} score={sc}{flag}")
print(f"\n  Tiers: .gov/journals > news > industry > social > noise")


</details>


---
## Exercise 5 (Medium): Citation Verification Agent



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)

sources={"C1":"GenAI course: 14999 INR, 146 hours, 14 modules",
    "C2":"GenAI salaries: 15-40 LPA, 300% demand growth",
    "C3":"Alternative: $294 (24500 INR), 6 months",
    "C4":"Student reviews: practical projects, good live classes"}

claims=[("Course costs 14999 INR, 14 modules","C1",True),("Salaries 15-40 LPA, 300% growth","C2",True),
    ("Cheaper than Coursera 24500 INR","C3",True),("95% job placement rate","C4",False),
    ("Includes hands-on GCP projects","C1",False),("Live classes help per reviews","C4",True)]

print("Citation Verification Agent:")
results={"supported":0,"partial":0,"unsupported":0}
for text,cite,expected in claims:
    src_words=set(sources.get(cite,"").lower().split())
    claim_words=set(text.lower().split())
    overlap=len(claim_words&src_words)/max(len(claim_words),1)
    verdict="supported" if overlap>0.4 else "partial" if overlap>0.2 else "unsupported"
    results[verdict]+=1
    print(f"  [{cite}] {verdict:>11}: {text[:45]}")

total=len(claims); acc=results["supported"]/total
print(f"\n  Supported:{results['supported']} Partial:{results['partial']} Unsupported:{results['unsupported']} Accuracy:{acc:.0%}")
print(f"  Industry: 50-90% of LLM citations don\'t support claims. Fix: CitationAgent + verification")


</details>


---
## Exercise 6 (Challenge): Full Deep Research Agent
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Full Deep Research Agent (LangGraph):")
print("  Pipeline: [decompose] -> Send() parallel per sub-query:")
print("    [search] -> [evaluate] -> [extract] per branch")
print("  -> [cross_check] -> [gap_analysis] -> conditional:")
print("     gaps? -> [re_plan] -> [search_more] | done? -> [synthesize] -> [cite]")
print("  4 architectures: OpenAI(o3,30-60 searches) | Google(visible plan,100+src)")
print("    Anthropic(multi-agent,90% better) | Perplexity(fastest,2-4min)")


</details>


---
## Exercise 7 (Challenge): Semantic Caching Layer
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)

class Cache:
    def __init__(self,th=0.8): self.th=th;self.c=[];self.h=0;self.m=0
    def _embed(self,t):
        v=np.zeros(50)
        for w in t.lower().split(): v[hash(w)%50]=1.0
        return v/(np.linalg.norm(v)+1e-8)
    def get(self,q):
        qv=self._embed(q)
        for e in self.c:
            sim=float(np.dot(qv,e["v"]))
            if sim>self.th: self.h+=1; return {"hit":True,"sim":round(sim,2),"cached":e["q"]}
        self.m+=1; return {"hit":False}
    def put(self,q,r): self.c.append({"q":q,"v":self._embed(q),"r":r})

cache=Cache(0.8)
queries=[("GenAI course price?","14999"),("GenAI course cost?",None),("Python details","9999"),
    ("GenAI pricing info",None),("Career prospects GenAI","15-40 LPA"),("GenAI career opportunities",None),
    ("Cloud courses","11999"),("Is GenAI worth the price?",None)]

print("Semantic Caching:")
for q,r in queries:
    c=cache.get(q)
    if c["hit"]: print(f"  HIT ({c['sim']}): '{q[:28]}' -> from '{c['cached'][:22]}'")
    else:
        if r: cache.put(q,r)
        print(f"  MISS: '{q[:28]}' -> search")
total=cache.h+cache.m; saved=cache.h*0.015
print(f"\n  Hits:{cache.h} Misses:{cache.m} Rate:{cache.h/total:.0%} Saved:${saved:.3f}")
print(f"  Industry: ~31% of queries are semantically similar")


</details>


---
## Exercise 8 (Challenge): Indian Language Research
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Indian Language Research:")
print("  Pipeline: Hindi query -> detect(hi) -> IndicTrans2 -> English")
print("  Search: English(Tavily/Serper) + Hindi(NewsData.io) + Govt(data.gov.in)")
print("  Synthesize: bilingual findings -> English report -> IndicTrans2 -> Hindi")
print("\n  Indian Data Sources:")
for n,d in [("data.gov.in","198K+ resources, API"),("RBI DBIE","Monetary data, API"),
    ("MoSPI eSankhyiki","GDP/CPI/IIP, MCP server"),("NewsData.io","13 Indian languages"),
    ("IRCTC/GST","Travel/tax, API preferred")]: print(f"  {n:<20} {d}")
print(f"\n  Cost: India Rs 0.50-4/query vs Global Rs 40-400/query")
print(f"  Monthly (1K/day): India Rs 15K-120K vs Global Rs 12L-120L")


</details>
